In [78]:
from databricks.connect import DatabricksSession

spark = DatabricksSession.builder.getOrCreate()

In [ ]:
catalog = dbutils.widgets.get("catalog")

In [88]:
import os
import sys

home = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))

sys.path.append(home)

from src.utils.json_parser import parse_json
from src.utils.clean_sales import good_sales_records, bad_sales_records
from src.utils.time_utils import date_day_weekOfMonth

In [ ]:
from pyspark.sql.types import StructField, IntegerType, StringType, StructType, TimestampType, LongType

# define schema for sales
schema = StructType([
    StructField("event_id", StringType(), True),
    StructField("sales_id", LongType(), True),
    StructField("employee_id", LongType(), True),
    StructField("region_id", IntegerType(), True),
    StructField("product_id", IntegerType(), True),
    StructField("quantity", IntegerType(), True),
    StructField("sales_amount", LongType(), True),
    StructField("event_time", TimestampType(), True),
    StructField("ingestion_time", TimestampType(), True)
])

df = spark.readStream.table(f"{catalog}.brz.sales_raw")

parsed_df = parse_json(spark, df, "value", schema)

parsed_df = (parsed_df.select("parsed.sales_id", "parsed.employee_id", "parsed.region_id",
                              "parsed.product_id", "parsed.quantity", "parsed.sales_amount",
                              "parsed.event_time"))

good_sales_df = good_sales_records(spark, parsed_df)

bad_sales_df = bad_sales_records(spark, parsed_df)

good_sales_df = date_day_weekOfMonth(spark, good_sales_df, "event_time")

good_sales_df = (good_sales_df.select("sales_id", "employee_id", "region_id", "product_id",
                                      "quantity", "sales_amount", "event_time", "event_date",
                                      "day", "week_of_month", "processed_time"))

good_query = (good_sales_df.writeStream
              .format("delta")
              .option("checkpointLocation", "abfss://checkpoints@jayveeradlsdevtest.dfs.core.windows.net/dev_checkpoints/slv_checkpoints/sales_checkpoint")
              .partitionBy("event_date")
              .outputMode("append")
              .trigger(availableNow = True)
              .table(f"{catalog}.slv.sales"))

bad_query = (bad_sales_df.writeStream
              .format("delta")
              .option("checkpointLocation", "abfss://checkpoints@jayveeradlsdevtest.dfs.core.windows.net/dev_checkpoints/brz_checkpoints/bad_sales_records_checkpoint")
              .outputMode("append")
              .trigger(availableNow = True)
              .table(f"{catalog}.brz.bad_sales_records"))

good_query.awaitTermination()
bad_query.awaitTermination()
